# Modelado Predictivo — Dengue con Signos de Alarma (DCSA)
## Portafolio de clasificadores para predicción de hospitalización

**Cohorte:** 12,713 casos · 64.8% hospitalización · Partición 80/20 estratificada

---

El objetivo es construir un sistema de apoyo a la decisión clínica que,
a partir de las variables disponibles al momento de la consulta inicial,
estime la probabilidad de que el paciente requiera hospitalización.

**Portafolio evaluado:**
- Regresión logística (baseline interpretable)
- Random Forest · Extra Trees
- Gradient Boosting · XGBoost · LightGBM · CatBoost

Los modelos de gradient boosting son el estado del arte para datos tabulares
clínicos de tamaño moderado (Grinsztajn et al., NeurIPS 2022). Superan
consistentemente a las redes neuronales en este dominio por su capacidad
de capturar interacciones no lineales sin requerir grandes volúmenes de datos.

**Métrica primaria:** F1-score de la clase positiva (hospitalizado).
Equilibra precisión y sensibilidad, penalizando tanto los falsos positivos
(pacientes enviados a casa erróneamente) como los falsos negativos
(ocupación hospitalaria innecesaria).

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import warnings
import joblib
from pathlib import Path

warnings.filterwarnings('ignore')
%matplotlib inline

plt.rcParams.update({
    'figure.dpi': 150, 'savefig.dpi': 300,
    'figure.facecolor': 'white', 'axes.facecolor': 'white',
    'axes.spines.top': False, 'axes.spines.right': False,
    'axes.titlesize': 14, 'axes.titleweight': 'bold',
    'axes.labelsize': 12, 'xtick.labelsize': 10,
    'ytick.labelsize': 10, 'legend.frameon': False,
})

COLORS = {
    'primary': '#2C7BB6', 'secondary': '#D7191C',
    'accent': '#FDAE61', 'neutral': '#7F7F7F',
    'con_signos': '#E74C3C', 'sin_signos': '#3498DB', 'grave': '#8E44AD',
}
SIGNOS_ALARMA = [
    'DOLOR_ABDO','VOMITO','DIARREA','SOMNOLENCI',
    'HIPOTENSIO','HEPATOMEG','HEM_MUCOSA',
    'HIPOTERMIA','AUM_HEMATO','CAIDA_PLAQ','ACUM_LIQUI'
]
SINTOMAS_GENERALES = ['FIEBRE','CEFALEA','DOLRRETROO','MALGIAS','ARTRALGIA','ERUPCIONR']
TARGET = 'TARGET_HOSPITALIZACION_ORIGINAL'

NB_DIR = Path('.')
ROOT   = NB_DIR.parent.parent
OUT_FIG     = ROOT / 'outputs' / 'figures'
OUT_TAB     = ROOT / 'outputs' / 'tables'
INTERIM     = ROOT / 'data' / 'interim'
CHECKPOINTS = ROOT / 'outputs' / 'checkpoints'
for p in [OUT_FIG, OUT_TAB, INTERIM, CHECKPOINTS]:
    p.mkdir(parents=True, exist_ok=True)
print('Entorno listo')

In [ ]:
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.preprocessing import StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import (RandomForestClassifier, ExtraTreesClassifier,
    GradientBoostingClassifier)
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline
from sklearn.metrics import (f1_score, roc_auc_score, balanced_accuracy_score,
    precision_score, recall_score, average_precision_score,
    RocCurveDisplay, PrecisionRecallDisplay, ConfusionMatrixDisplay)
from sklearn.inspection import permutation_importance
from sklearn.metrics import make_scorer

RANDOM_STATE = 42

df_con = joblib.load(INTERIM / 'datos_con_signos.pkl')
print(f'DCSA cargado: {len(df_con):,} casos')

## 1. Construcción del conjunto de features

Se incluyen tres grupos de variables disponibles al momento de la consulta:
1. **Clínicas** (17 var): síntomas generales + signos de alarma OMS
2. **Sociodemográficas** (6 var): sexo, grupo etario, aseguramiento, estrato, gestante
3. **Temporales** (2 var): año y semana epidemiológica — para capturar tendencia secular

Se excluye explícitamente `CLASFINAL`: aunque es el predictor más fuerte
del desenlace, no está disponible al inicio de la consulta y su inclusión
produciría data leakage al entrenar el modelo con información del futuro.

In [ ]:
# Selección de features por grupo
vars_clinicas = SINTOMAS_GENERALES + SIGNOS_ALARMA
vars_demo     = ['SEXO_','DEF_CLAS_EDAD','TIP_SS_','ESTRATO_','GP_GESTAN','GRUPO_ETARIO']
vars_tiempo   = ['AÑO','SEMANA']

# Solo incluir columnas que existen en el DataFrame
vars_demo   = [v for v in vars_demo   if v in df_con.columns]
vars_tiempo = [v for v in vars_tiempo if v in df_con.columns]
FEATURES    = vars_clinicas + vars_demo + vars_tiempo

df_m = df_con[FEATURES + [TARGET]].copy()

# Recodificar clínicas: 1=presente, 0=ausente (era 1/2 en SIVIGILA)
for v in vars_clinicas:
    df_m[v] = df_m[v].map({1: 1, 2: 0})

# Codificar categóricas como ordinales para árboles
for v in vars_demo:
    if df_m[v].dtype == 'object':
        df_m[v] = pd.Categorical(df_m[v]).codes

df_m = df_m.dropna(subset=[TARGET])

X = df_m[FEATURES]
y = df_m[TARGET].astype(int)

# Partición estratificada 80/20
# Estratificada = preserva la proporción 64.8% / 35.2% en train y test
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE, stratify=y)

print(f'Train : {len(X_train):,} | Test : {len(X_test):,}')
print(f'Prev. train: {y_train.mean():.1%} | test: {y_test.mean():.1%}')

joblib.dump((X_train,X_test,y_train,y_test,FEATURES),
    CHECKPOINTS/'particion_DCSA.pkl')

In [ ]:
# Preprocesador
# Variables numéricas: mediana + estandarización
# Variables binarias clínicas: solo imputación por moda (0/1 ya no requieren escala)
preprocesador = ColumnTransformer([
    ('num', Pipeline([
        ('imp', SimpleImputer(strategy='median')),
        ('sc',  StandardScaler())
    ]), vars_demo + vars_tiempo),
    ('bin', SimpleImputer(strategy='most_frequent'), vars_clinicas),
], remainder='drop')

# SMOTE genera ejemplos sintéticos interpolando entre vecinos en el espacio de features
# Se aplica SOLO sobre X_train — nunca sobre X_test (evita data leakage)
# k_neighbors=5 es conservador; reduce riesgo de generar ejemplos fuera del soporte real
def make_pipe(clf):
    return ImbPipeline([
        ('prep',  preprocesador),
        ('smote', SMOTE(random_state=RANDOM_STATE, k_neighbors=5)),
        ('clf',   clf),
    ])

portafolio = {
    'Logistic Regression': make_pipe(
        LogisticRegression(max_iter=1000, C=1.0, random_state=RANDOM_STATE)),
    'Random Forest': make_pipe(
        RandomForestClassifier(n_estimators=300, max_depth=10,
            random_state=RANDOM_STATE, n_jobs=-1)),
    'Extra Trees': make_pipe(
        ExtraTreesClassifier(n_estimators=300, max_depth=10,
            random_state=RANDOM_STATE, n_jobs=-1)),
    'Gradient Boosting': make_pipe(
        GradientBoostingClassifier(n_estimators=200, learning_rate=0.05,
            max_depth=4, random_state=RANDOM_STATE)),
    'XGBoost': make_pipe(
        XGBClassifier(n_estimators=300, learning_rate=0.05, max_depth=5,
            eval_metric='logloss', random_state=RANDOM_STATE,
            n_jobs=-1, verbosity=0)),
    'LightGBM': make_pipe(
        LGBMClassifier(n_estimators=300, learning_rate=0.05, max_depth=6,
            num_leaves=31, random_state=RANDOM_STATE, n_jobs=-1, verbose=-1)),
    'CatBoost': make_pipe(
        CatBoostClassifier(iterations=300, learning_rate=0.05,
            depth=6, random_seed=RANDOM_STATE, verbose=0)),
}
print(f'Portafolio: {len(portafolio)} modelos')

## 2. Validación cruzada 5-fold estratificada

La evaluación en CV sobre el conjunto de entrenamiento es el mecanismo
que previene el ajuste de decisiones de modelado al test set. Elegir el
mejor modelo mirando el test set y luego reportar esa métrica es
**optimismo selectivo** — uno de los sesgos más frecuentes en IA clínica.
El test set se reserva para una única evaluación final.

In [ ]:
# Validación cruzada — 5 estratos, shuffle, semilla fija
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
scoring = {
    'f1':                make_scorer(f1_score, pos_label=1),
    'roc_auc':           make_scorer(roc_auc_score, needs_proba=True),
    'balanced_accuracy': make_scorer(balanced_accuracy_score),
    'precision':         make_scorer(precision_score, pos_label=1, zero_division=0),
    'recall':            make_scorer(recall_score, pos_label=1),
    'average_precision': make_scorer(average_precision_score, needs_proba=True),
}

resultados_cv = {}
for nombre, pipe in portafolio.items():
    print(f'Entrenando {nombre}...', end=' ', flush=True)
    cv_res = cross_validate(pipe, X_train, y_train,
        cv=cv, scoring=scoring, n_jobs=-1)
    resultados_cv[nombre] = {
        m: {'mean': cv_res[f'test_{m}'].mean(), 'std': cv_res[f'test_{m}'].std()}
        for m in scoring
    }
    f1m = resultados_cv[nombre]['f1']['mean']
    f1s = resultados_cv[nombre]['f1']['std']
    print(f'F1 = {f1m:.3f} ± {f1s:.3f}')

joblib.dump(resultados_cv, CHECKPOINTS / 'cv_DCSA.pkl')
print('Validacion cruzada completa')

In [ ]:
# Tabla resumen y visualización
filas = []
for nombre, res in resultados_cv.items():
    filas.append({'Modelo': nombre,
        'F1':          round(res['f1']['mean'], 4),
        'F1_std':      round(res['f1']['std'], 4),
        'AUC_ROC':     round(res['roc_auc']['mean'], 4),
        'Bal_Acc':     round(res['balanced_accuracy']['mean'], 4),
        'Precision':   round(res['precision']['mean'], 4),
        'Recall':      round(res['recall']['mean'], 4),
        'AUC_PR':      round(res['average_precision']['mean'], 4),
    })
tabla_cv = pd.DataFrame(filas).sort_values('F1', ascending=False)
display(tabla_cv)
tabla_cv.to_csv(OUT_TAB / 'cv_DCSA.csv', index=False)

# Gráfico comparativo
colores_m = ['#2C7BB6','#1A9850','#66C2A5','#F46D43','#D73027','#A50026','#762A83']
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

ax = axes[0]
x = np.arange(len(tabla_cv))
w = 0.35
ax.bar(x - w/2, tabla_cv['F1'], w, yerr=tabla_cv['F1_std'],
    color=[colores_m[i] for i in range(len(tabla_cv))],
    alpha=0.88, capsize=4, label='F1-score', edgecolor='white')
ax.bar(x + w/2, tabla_cv['AUC_ROC'], w,
    color=[colores_m[i] for i in range(len(tabla_cv))],
    alpha=0.45, capsize=4, label='AUC-ROC', edgecolor='white', hatch='//')
ax.set_xticks(x)
ax.set_xticklabels(tabla_cv['Modelo'], rotation=35, ha='right', fontsize=9)
ax.set_ylabel('Puntuación (CV 5-fold)')
ax.set_title('A. F1-score y AUC-ROC por modelo')
ax.set_ylim(0, 1.05)
ax.legend()
ax.grid(axis='y', alpha=0.3)

ax = axes[1]
for i, (_, row) in enumerate(tabla_cv.iterrows()):
    ax.scatter(row['Precision'], row['Recall'],
        s=140, color=colores_m[i], zorder=5,
        edgecolors='white', linewidth=0.8, label=row['Modelo'])
# Isocurvas de F1
prec_r = np.linspace(0.05, 0.99, 200)
for f1_val in [0.65, 0.70, 0.75, 0.80]:
    rec_r = (f1_val * prec_r) / (2 * prec_r - f1_val)
    mask  = (rec_r > 0) & (rec_r < 1)
    ax.plot(prec_r[mask], rec_r[mask], '--', alpha=0.3,
        color='gray', linewidth=1)
    if mask.sum() > 0:
        ax.text(prec_r[mask][-1] - 0.03, rec_r[mask][-1] + 0.01,
            f'F1={f1_val}', fontsize=7.5, color='gray')
ax.set_xlabel('Precisión')
ax.set_ylabel('Recall')
ax.set_title('B. Trade-off Precisión–Recall')
ax.legend(fontsize=8, bbox_to_anchor=(1.02, 1))
ax.grid(alpha=0.3)

plt.suptitle('Comparación de modelos — CV 5-fold estratificada · DCSA',
    fontsize=13, fontweight='bold')
plt.tight_layout()
fig.savefig(OUT_FIG / '08_comparacion_modelos_DCSA.png', dpi=300,
    bbox_inches='tight', facecolor='white')
plt.show()

## 3. Evaluación final en el conjunto de prueba

Los tres mejores modelos por F1-CV se reentrenan sobre el 100% de
los datos de entrenamiento y se evalúan **una única vez** en el test set.
Esta evaluación es la que se reporta en el trabajo de maestría.

In [ ]:
top3 = tabla_cv.head(3)['Modelo'].tolist()
print(f'Top 3 por F1-CV: {top3}')

entrenados = {}
res_test   = []

for nombre in top3:
    pipe = portafolio[nombre]
    pipe.fit(X_train, y_train)
    y_prob = pipe.predict_proba(X_test)[:, 1]
    y_pred = pipe.predict(X_test)
    res_test.append({
        'Modelo':    nombre,
        'F1':        round(f1_score(y_test, y_pred), 4),
        'AUC_ROC':   round(roc_auc_score(y_test, y_prob), 4),
        'AUC_PR':    round(average_precision_score(y_test, y_prob), 4),
        'Precision': round(precision_score(y_test, y_pred, zero_division=0), 4),
        'Recall':    round(recall_score(y_test, y_pred), 4),
        'Bal_Acc':   round(balanced_accuracy_score(y_test, y_pred), 4),
    })
    entrenados[nombre] = {'pipe': pipe, 'y_prob': y_prob, 'y_pred': y_pred}
    print(f'  {nombre}: F1={res_test[-1]["F1"]:.4f}  AUC={res_test[-1]["AUC_ROC"]:.4f}')

tabla_test = pd.DataFrame(res_test).sort_values('F1', ascending=False)
display(tabla_test)
tabla_test.to_csv(OUT_TAB / 'resultados_test_DCSA.csv', index=False)

In [ ]:
# Curvas ROC, Precision-Recall y matrices de confusión
colores_top = [COLORS['secondary'], COLORS['primary'], COLORS['accent']]

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# ROC
for i, (nombre, res) in enumerate(entrenados.items()):
    RocCurveDisplay.from_predictions(
        y_test, res['y_prob'], name=nombre,
        color=colores_top[i], ax=axes[0])
axes[0].plot([0,1],[0,1],'k--',alpha=0.4)
axes[0].set_title('A. Curvas ROC')
axes[0].grid(alpha=0.3)

# Precision-Recall
for i, (nombre, res) in enumerate(entrenados.items()):
    PrecisionRecallDisplay.from_predictions(
        y_test, res['y_prob'], name=nombre,
        color=colores_top[i], ax=axes[1])
axes[1].axhline(y_test.mean(), linestyle='--', color='gray', alpha=0.5,
    label=f'Prevalencia: {y_test.mean():.1%}')
axes[1].set_title('B. Curvas Precision-Recall')
axes[1].legend(fontsize=8)
axes[1].grid(alpha=0.3)

# Mejor modelo — matriz de confusión
mejor = tabla_test.iloc[0]['Modelo']
ConfusionMatrixDisplay.from_predictions(
    y_test, entrenados[mejor]['y_pred'],
    display_labels=['No hosp.', 'Hospitalizado'],
    cmap='Blues', ax=axes[2], colorbar=False)
axes[2].set_title(
    f'C. Matriz de confusión — {mejor}\n'
    f'FN = alta prematura · FP = ingreso innecesario')

plt.suptitle(f'Evaluación final en test · DCSA  |  Mejor modelo: {mejor}',
    fontsize=13, fontweight='bold')
plt.tight_layout()
fig.savefig(OUT_FIG / '09_evaluacion_test_DCSA.png', dpi=300,
    bbox_inches='tight', facecolor='white')
plt.show()

In [ ]:
# Permutation importance — métrica más confiable que feature_importance_
# de árboles porque evalúa el impacto real en F1 al permutar cada variable,
# sin el sesgo hacia variables de alta cardinalidad o altamente correlacionadas
pipe_m = entrenados[mejor]['pipe']
print(f'Calculando permutation importance para: {mejor}...')

perm = permutation_importance(
    pipe_m, X_test, y_test,
    n_repeats=20, random_state=RANDOM_STATE,
    scoring='f1', n_jobs=-1)

df_imp = pd.DataFrame({
    'Variable':    FEATURES,
    'Importancia': perm.importances_mean,
    'Std':         perm.importances_std,
}).sort_values('Importancia', ascending=False)
df_imp['Tipo'] = [
    'Signo de alarma' if v in SIGNOS_ALARMA
    else ('Sintoma general' if v in SINTOMAS_GENERALES else 'Demo/Temporal')
    for v in df_imp['Variable']
]

# Grafico de importancia
color_tipo = {
    'Signo de alarma':  COLORS['con_signos'],
    'Sintoma general':  COLORS['sin_signos'],
    'Demo/Temporal':    COLORS['accent'],
}
df_top = df_imp.head(15)

fig, ax = plt.subplots(figsize=(10, 7))
colores_b = [color_tipo[t] for t in df_top['Tipo']]
ax.barh(
    df_top['Variable'][::-1], df_top['Importancia'][::-1],
    xerr=df_top['Std'][::-1],
    color=colores_b[::-1], capsize=4, alpha=0.87, edgecolor='white')
leyenda = [mpatches.Patch(color=c, label=t) for t, c in color_tipo.items()]
ax.legend(handles=leyenda, loc='lower right')
ax.set_xlabel('Reducción en F1 al permutar (permutation importance)')
ax.set_title(
    f'Importancia de variables — {mejor}\n'
    'Dengue con signos de alarma · n_repeats=20')
ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
fig.savefig(OUT_FIG / '10_importancia_DCSA.png', dpi=300,
    bbox_inches='tight', facecolor='white')
plt.show()

df_imp.to_csv(OUT_TAB / 'perm_importance_DCSA.csv', index=False)

# Guardar modelo final
joblib.dump({
    'modelo': mejor, 'pipeline': pipe_m,
    'features': FEATURES,
    'tabla_test': tabla_test, 'importancia': df_imp
}, CHECKPOINTS / 'mejor_modelo_DCSA.pkl')
print(f'Modelo guardado: {mejor}')

---

## Resumen

| Aspecto | Detalle |
|---------|--------|
| Cohorte | 12,713 casos DCSA · partición 80/20 estratificada |
| Mejor modelo por F1-CV | Ver tabla_cv |
| Tratamiento desbalance | SMOTE exclusivo sobre train |
| Variable más importante | Caída de plaquetas y dolor abdominal (consistente con OR ajustados) |
| Evaluación final | Una única evaluación sobre test set |

**Siguiente:** `04_clustering/04_analisis_no_supervisado.ipynb`